## MLegS Post-Processing Visualization

This notebook is designed to read and visualize 2D field data generated by the `postproc` executable from the MLegS simulation package.

### Functionality:
1.  **Reads Collocation Points**: Loads the physical grid coordinates (`r`, `theta`, `z`) from the `.info` files in the output directory.
2.  **Identifies Data Files**: Scans the output directory for 2D slice data files (e.g., `velR_RTplane_001.dat`, `vorZ_RZplane_010.dat`).
3.  **Parses Filenames**: Extracts metadata from filenames, such as the field type, slice orientation, and snapshot index.
4.  **Loads and Reshapes Data**: Reads the raw data, which is stored as pairs of real and imaginary components, and reshapes it into the correct 2D complex array corresponding to the slice.
5.  **Visualizes Fields**: Creates contour plots for the real part of the selected field data. It handles both R-Theta and R-Z plane visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from ipywidgets import Dropdown, FloatSlider, Checkbox, FloatText, HBox, VBox, Label, interactive_output
from IPython.display import display

## Load data

In [ ]:
def load_collocation_points(directory):
    """Loads r, theta, and z grid points from .info files."""
    try:
        r_pts = np.loadtxt(os.path.join(directory, 'r_colloc_pts.info'))
        th_pts = np.loadtxt(os.path.join(directory, 't_colloc_pts.info'))
        z_pts = np.loadtxt(os.path.join(directory, 'z_colloc_pts.info'))
        print(f"Loaded grid points:")
        print(f"  - Radial (r): {len(r_pts)} points")
        print(f"  - Azimuthal (theta): {len(th_pts)} points")
        print(f"  - Axial (z): {len(z_pts)} points")
        return r_pts, th_pts, z_pts
    except FileNotFoundError as e:
        print(f"Error loading grid files: {e}")
        print("Please ensure 'r_colloc_pts.info', 't_colloc_pts.info', and 'z_colloc_pts.info' are in the output directory.")
        return None, None, None
    
def find_data_files(directory):
    """Finds and parses 2D slice data files in the given directory."""
    # Regex to match filenames like 'velR_RTplane_001.dat' or 'vorZ_RZplane_010.dat'
    # It captures the field name, slice type, and index.
    # It captures the field name, slice type, and index (either 3 digits or 'ini').
    pattern = re.compile(r"^(?P<field>\w+?)_(?P<slice_type>RTplane|RZplane)_(?P<index>\d{3}|ini)\.dat$")
    
    file_info = []
    print(f"\nScanning for data files in: {directory}")
    for filename in sorted(os.listdir(directory)):
        match = pattern.match(filename)
        if match:
            info = match.groupdict()
            info['filename'] = filename
            file_info.append(info)
            
    if not file_info:
        print("No 2D slice data files found. Make sure 'POSTPROCESS%SLICEINT' is not 999 and filenames match the expected pattern.")
    else:
        print(f"Found {len(file_info)} data files.")
        for info in file_info:
            print(f"  - {info['filename']}: Field = {info['field']}, Slice = {info['slice_type']}, Index = {info['index']}")
        
    return file_info

def load_and_reshape_data(filepath, slice_type, grid_shapes):
    """
    Loads data from a file and reshapes it into the correct 2D real-valued array.
    - For RTplane, the data is real across all theta points.
    - For RZplane, the z-direction is periodic. The data is saved for NZ points,
      but the coordinates have NZ+1 points. We pad the data array by copying
      the first z-slice to the end to close the domain for plotting.
      
    Data storage convention:
    - For 2D data: slowest dimension is radial (r), fastest is the other dimension
    - For 3D data: slowest is z, then r, then theta (fastest)
    """
    try:
        # The data is a single line of text with space-separated numbers.
        raw_data = np.loadtxt(filepath).T
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

    # Determine the correct shape based on the slice type
    nr, nth, nz = grid_shapes
    if slice_type == 'RTplane':
        # For RT plane: data layout is nr x (nth-1) with radial as slowest dimension
        # The first nth values are for r=0, next nth for r=1, etc.
        expected_shape = (nth - 1, nr)  # Shape of raw data as stored
        real_data_unpadded = raw_data
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :]))  # Pad theta by appending first column

        return real_data.T
        
    elif slice_type == 'RZplane':
        # For RZ plane: data layout is nr x (nz-1) with radial as slowest dimension
        # The first (nz-1) values are for r=0, next (nz-1) for r=1, etc.
        expected_shape = (nz - 1, nr)  # Shape of complex data as stored
        
        if raw_data.size % 2 != 0:
            print(f"Warning: RZ data in {filepath} has an odd number of elements.")
            return None
            
        # Reconstruct complex numbers and take the real part for the slice.
        complex_data = raw_data[::2] + 1j * raw_data[1::2]
        real_data_unpadded = np.real(complex_data)
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :])) # Pad z by appending first column

        return real_data.T # Return padded data with shape (nr, nz)
    else:
        return None


## Visualization

In [ ]:
def find_roots(r, f):
    """
    Finds the roots of a function f(r) using linear interpolation
    where sign changes are detected.
    """
    roots = []
    # Find indices *before* a sign change
    sign_changes = np.where(np.diff(np.sign(f)))[0]
    
    for i in sign_changes:
        # Check for non-finite values (like NaN from sqrt of negatives)
        if not np.isfinite(f[i]) or not np.isfinite(f[i+1]):
            continue
            
        r1, r2 = r[i], r[i+1]
        f1, f2 = f[i], f[i+1]
        
        # Linear interpolation formula to find r_root where f=0
        r_root = r1 - f1 * (r2 - r1) / (f2 - f1)
        roots.append(r_root)
        
    return roots

def find_critical_radii(m, filepath, N):
    """
    Finds all critical radii for a given m by reading a baseflow file.
    
    Returns:
    --------
    dict or None:
        A dictionary containing two lists: 'N_roots' and 'Delta_roots'.
        Returns None if the file cannot be read.
    """
    
    if not os.path.exists(filepath):
        print(f"Warning: Baseflow file not found at '{filepath}'. Cannot plot critical layers.")
        return None

    try:
        # Assumes file format: r, ut, oz, omega0, rayleigh_discriminant
        data = np.loadtxt(filepath)
        r = data[:, 0]
        omega0 = data[:, 3] # Angular velocity
        delta = data[:, 4]   # Rayleigh discriminant
    except Exception as e:
        print(f"Warning: Failed to load or parse baseflow file '{filepath}'. Error: {e}")
        return None

    # m=0 has no critical layers (as m*Omega = 0)
    # We also handle the case where m=0 and N=0, which is ill-defined.
    if m == 0:
        return {'N_roots': [], 'Delta_roots': []}

    # --- Find N roots ---
    f_N_pos = m * omega0 - N
    f_N_neg = m * omega0 + N
    crit_r_N_pos = find_roots(r, f_N_pos)
    crit_r_N_neg = find_roots(r, f_N_neg)
    all_N_roots = crit_r_N_pos + crit_r_N_neg

    # --- Find Delta roots ---
    sqrt_delta = np.full_like(delta, np.nan)
    stable_mask = delta >= 0
    sqrt_delta[stable_mask] = np.sqrt(delta[stable_mask])

    f_D_pos = m * omega0 - sqrt_delta
    f_D_neg = m * omega0 + sqrt_delta
    crit_r_D_pos = find_roots(r, f_D_pos)
    crit_r_D_neg = find_roots(r, f_D_neg)
    all_Delta_roots = crit_r_D_pos + crit_r_D_neg
    
    return {
        'N_roots': all_N_roots,
        'Delta_roots': all_Delta_roots
    }

In [ ]:
def plot_placeholder(ax, title):
    """Plots a placeholder on the given axis."""
    ax.set_title(title, va='bottom', fontsize=14)
    ax.text(0.5, 0.5, "Data Not Available", 
            horizontalalignment='center', 
            verticalalignment='center', 
            transform=ax.transAxes, 
            fontsize=12, color='gray',
            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.5'))
    
    # Clear ticks and labels for a cleaner look
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Special handling for polar plots
    if hasattr(ax, 'set_rgrids'):
         ax.set_rgrids([])
         ax.set_thetagrids([])
    return

def plot_rt_slice(ax, data, r_coords, th_coords, title, r_min = 0.0, r_max = 5.0, use_log_scale=True, 
                  mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None, f_eff=None):
    """Creates a polar contour plot on a given ax.
    
    Parameters:
    -----------
    ax : matplotlib.axes.Axes
        The polar subplot axis to draw on.
    data : np.ndarray or None
        The 2D data to plot. If None, a placeholder is drawn.
    ... (other parameters) ...
    """
    if data is None:
        plot_placeholder(ax, title + "\n(File not found or failed to load)")
        return
    
    if f_eff is not None:
        if f_eff.shape == r_coords.shape:
            # Handle potential divide-by-zero
            f_eff_safe = np.where(np.abs(f_eff) < 1e-16, 1e-16, f_eff)
            # Use broadcasting to divide data[i, j] by f_eff_safe[i]
            data_norm = data / f_eff_safe[:, np.newaxis]
            title += " normalized w/ $f_{eff}$"
        else:
            print(f"Warning (rt_slice): f_eff shape ({f_eff.shape}) does not match r_coords shape ({r_coords.shape}). Skipping normalization.")
            return
    else:
        data_norm = data

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data_norm[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Create a meshgrid for polar plot
    R, TH = np.meshgrid(r_filtered, th_coords)

    # Get the parent figure
    fig = ax.get_figure()
    
    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        max_log_val = np.nanmax(np.abs(data_log)) # Use nanmax for masked arrays
        vmin, vmax = -max_log_val, max_log_val
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        contour = ax.pcolormesh(TH, R, data_log, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        
        # Add colorbar, shrink to fit
        cbar = fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value', shrink=0.8)
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
    else:
        if max_abs_val < 1e-16:
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        contour = ax.pcolormesh(TH, R, data_filtered_masked.T, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        # Add colorbar, shrink to fit
        fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value', shrink=0.8)
    
    # Add reference line at r = ref_r if specified
    if ref_r is None:
        ref_r = r_coords[r_coords.size * 2 // 3]
    ax.plot(th_coords, np.full_like(th_coords, ref_r), color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
    ax.legend(loc='upper right')

    ax.set_title(title, va='bottom', fontsize=14)
    ax.set_ylim(0, r_max) # Set radial limit to r_max
    ax.set_xlabel('$\\theta$')
    ax.set_ylabel('$r$', labelpad=20)

def plot_rz_slice(ax, data, r_coords, z_coords, title, r_min=0.0, r_max=5.0, use_log_scale=True,
                  mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None, f_eff=None):
    """Creates a Cartesian contour plot on a given ax."""
    if data is None:
        plot_placeholder(ax, title + "\n(File not found or failed to load)")
        return
    
    plot_r_ind=False

    if f_eff is not None:
        if f_eff.shape == r_coords.shape:
            # Handle potential divide-by-zero
            f_eff_safe = np.where(np.abs(f_eff) < 1e-16, 1e-16, f_eff)
            # Use broadcasting to divide data[i, j] by f_eff_safe[i]
            data_norm = data / f_eff_safe[:, np.newaxis]
            title += " normalized w/ $f_{eff}$"
        else:
            print(f"Warning (rz_slice): f_eff shape ({f_eff.shape}) does not match r_coords shape ({r_coords.shape}). Skipping normalization.")
            return
    else:
        data_norm = data

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data_norm[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Calculate ZLEN for normalization
    ZLEN = np.max(z_coords) - np.min(z_coords)
    z_normalized = z_coords / ZLEN
    
    # Get the parent figure
    fig = ax.get_figure()

    # Determine the horizontal axis
    if plot_r_ind:
        x_axis = np.arange(len(r_filtered))
        x_label = 'Radial Index'
    else:
        x_axis = r_filtered
        x_label = 'Radial Coordinate (r)'

    # Create a meshgrid for the plot
    X, Z = np.meshgrid(x_axis, z_normalized)

    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.nanmax(np.abs(data_log)) # Use nanmax for masked arrays
        vmin, vmax = -max_log_val, max_log_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(X, Z, data_log, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, label='Field Value', shrink=0.8)
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        if max_abs_val < 1e-16:
            # For all-zero data, use small symmetric limits
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(X, Z, data_filtered_masked.T, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        # Add colorbar, shrink to fit
        fig.colorbar(contour, ax=ax, label='Field Value', shrink=0.8)

    if ref_r is None:
        ref_r = r_coords[r_coords.size * 2 // 3]
    if plot_r_ind:
        ref_r_ind = np.argmin(np.abs(r_filtered - ref_r))
        ax.axvline(x=ref_r_ind, color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
        n = 0
        while n < len(r_filtered):
            n += 13
            ax.axvline(x=n, color='red', linestyle=':', linewidth=.5)
    else:
        ax.axvline(x=ref_r, color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
    ax.legend(loc='upper right')

    ax.set_title(title, fontsize=14)
    ax.set_xlabel(x_label)
    ax.set_ylabel('Axial Coordinate (z/ZLEN)')
    if not plot_r_ind:
        ax.set_xlim(0, r_max) # Set radial limit to r_max
    else:
        ax.set_xlim(0, len(r_filtered)-1)

def interactive_plotter(field_index_str, r_min = 0.0, r_max=5.0, use_log_scale=False, rt_use_log_scale=False, rz_use_log_scale=False,
                       mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None):
    """
    Main function driven by the ipywidgets interact decorator.
    Plots RT and RZ slices side-by-side for the selected file's index.
    """
    if not field_index_str or not all((r_pts is not None, th_pts is not None, z_pts is not None)):
        print("Cannot plot. Check that data and grid files were loaded correctly or select a field.")
        return
    
    # Calculate f_eff if baseflow file exists
    N_val = 0.07
    Omega = 0.0  # Background (constant) rotation rate
    baseflow_filepath = '../../output/baseflow.dat'
    f_eff = None
    if os.path.exists(baseflow_filepath):
        try:
            # Assumes file format: r, ut, oz, omega0, rayleigh_discriminant
            base_data = np.loadtxt(baseflow_filepath)
            r_base = base_data[:, 0]
            omega0 = base_data[:, 3] # \bar{\Omega}(r)
            
            # Check if baseflow r-grid (r_base) matches plot r-grid (r_pts)
            if np.array_equal(r_base, r_pts):
                f_eff = 2 * (Omega + omega0)
            else:
                # If grids don't match, interpolate f_eff onto the plot grid (r_pts)
                print(f"Warning: Baseflow r-grid (size {len(r_base)}) does not match plot r-grid (size {len(r_pts)}). Interpolating f_eff.")
                f_eff = np.interp(r_pts, r_base, 2 * (Omega + omega0))
                
        except Exception as e:
            print(f"Warning: Failed to load or parse baseflow file '{baseflow_filepath}' for f_eff. Error: {e}")
    else:
        print(f"Warning: Baseflow file not found at '{baseflow_filepath}'. Cannot calculate f_eff.")

    # Parse the 'field_index_str' (e.g., "E_001")
    try:
        target_field, target_index = field_index_str.split('_', 1) # Split only on the first underscore
    except ValueError:
        print(f"Invalid selection format: {field_index_str}. Expected 'Field_Index'.")
        return

    grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

    # --- Find RT and RZ info for the same index and field ---
    rt_info = next((item for item in data_files if item['field'] == target_field and item['slice_type'] == 'RTplane' and item['index'] == target_index), None)
    rz_info = next((item for item in data_files if item['field'] == target_field and item['slice_type'] == 'RZplane' and item['index'] == target_index), None)

    # --- Load RT Data ---
    rt_data = None
    rt_title = f"Field: {target_field} | Slice: RTplane | Index: {target_index}"
    if rt_info:
        filepath_rt = os.path.join(output_dir, rt_info['filename'])
        rt_data = load_and_reshape_data(filepath_rt, 'RTplane', grid_shapes)
        if rt_data is None:
            print(f"Failed to load or process data for {rt_info['filename']}.")
    else:
        print(f"No RTplane file found for field {target_field}, index {target_index}")
    
    # --- Load RZ Data ---
    rz_data = None
    rz_title = f"Field: {target_field} | Slice: RZplane | Index: {target_index}"
    if rz_info:
        filepath_rz = os.path.join(output_dir, rz_info['filename'])
        rz_data = load_and_reshape_data(filepath_rz, 'RZplane', grid_shapes)
        if rz_data is None:
            print(f"Failed to load or process data for {rz_info['filename']}.")
    else:
         print(f"No RZplane file found for field {target_field}, index {target_index}")

    # --- Create the figure and axes ---
    fig = plt.figure(figsize=(18, 8)) # Wider figure for side-by-side
    ax1 = fig.add_subplot(1, 2, 1, projection='polar')
    ax2 = fig.add_subplot(1, 2, 2)

    # --- Plot RT Slice (or placeholder) ---
    plot_rt_slice(ax1, rt_data, r_pts, th_pts, rt_title, 
                 r_min=r_min, r_max=r_max, 
                 use_log_scale=rt_use_log_scale or use_log_scale,
                 mask_low_intensity=mask_low_intensity, 
                 intensity_threshold_orders=intensity_threshold_orders,
                 ref_r=ref_r, f_eff=f_eff)

    # --- Plot RZ Slice (or placeholder) ---
    plot_rz_slice(ax2, rz_data, r_pts, z_pts, rz_title, 
                 r_min=r_min, r_max=r_max, 
                 use_log_scale=rz_use_log_scale or use_log_scale,
                 mask_low_intensity=mask_low_intensity,
                 intensity_threshold_orders=intensity_threshold_orders,
                 ref_r=ref_r, f_eff=f_eff)
    
    # # Add a main title for the whole figure
    # fig.suptitle(f"Field: {target_field} | Index: {target_index}", fontsize=16, y=1.02)
    # plt.tight_layout()
    # plt.show() # Show the combined figure

    # Plot critical layers for m=0 to 9
    label_N_added = False
    label_Delta_added = False
    
    for m in range(10): # Loop m from 0 to 9
        critical_roots = find_critical_radii(m, baseflow_filepath, N_val)
        
        if not critical_roots:
            continue # Skips if file wasn't found
            
        # Plot N-related roots (m*Omega = +/- N)
        for r_val in critical_roots['N_roots']:
            if r_min <= r_val <= r_max:
                label = '|m*Omega| = N$' if not label_N_added else None
                
                # # Plot on ax1 (Polar)
                # ax1.plot(th_pts, np.full_like(th_pts, r_val), 
                #          color='r', linestyle='-', lw=1.5, label=label, alpha=0.5)
                
                # Plot on ax2 (Cartesian)
                ax2.axvline(x=r_val, color='r', linestyle='-', lw=1.5, label=label, alpha=0.5)
                
                label_N_added = True # Mark as added
        
        # Plot Delta-related roots (m*Omega = +/- sqrt(Delta))
        for r_val in critical_roots['Delta_roots']:
            if r_min <= r_val <= r_max:
                label = '|m*Omega| = ±sqrtDelta' if not label_Delta_added else None
                
                # # Plot on ax1 (Polar)
                # ax1.plot(th_pts, np.full_like(th_pts, r_val), 
                #          color='r', linestyle='--', lw=1.5, label=label, alpha=0.5)
                
                # Plot on ax2 (Cartesian)
                ax2.axvline(x=r_val, color='r', linestyle='--', lw=1.5, label=label, alpha=0.5)
                
                label_Delta_added = True # Mark as added

    # --- Re-build legends to include all plotted lines ---
    ax1.legend(loc='upper right', fontsize='small')
    ax2.legend(loc='upper right', fontsize='small')
    
    # Update title to show m=0-6 are included
    fig.suptitle(f"Field: {target_field} | Index: {target_index} (m=0-6)", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show() # Show the combined figure

## Analysis

In [ ]:
output_dir = '../../output/'
r_pts, th_pts, z_pts = load_collocation_points(output_dir)

data_files = find_data_files(output_dir)
info = data_files[0]
filepath = os.path.join(output_dir, info['filename'])
grid_shapes = (len(r_pts), len(th_pts), len(z_pts))
data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)

In [ ]:
if data_files:

    field_index_pairs = sorted(list(set((info['field'], info['index']) for info in data_files)))
    dropdown_options = [f"{field}_{index}" for field, index in field_index_pairs]
    
    # Create widgets
    field_index_dropdown = Dropdown(options=dropdown_options, description='Field/Index:')
    r_min_slider = FloatSlider(value=2.0, min=0.0, max=200.0, step=0.1, description='r_min:', continuous_update=False)
    r_max_slider = FloatSlider(value=30.0, min=1.0, max=200.0, step=0.1, description='r_max:', continuous_update=False)
    log_both_check = Checkbox(value=False, description='Log (both)')
    log_rt_check = Checkbox(value=True, description='Log RT')
    log_rz_check = Checkbox(value=True, description='Log RZ')
    mask_check = Checkbox(value=False, description='Mask low intensity')
    threshold_slider = FloatSlider(value=2.0, min=1.0, max=6.0, step=0.5, description='Orders:', continuous_update=False)
    # ref_r_input = FloatText(value=0.6, description='ref_r:', disabled=True)

    ui = VBox([
        field_index_dropdown,
        HBox([r_min_slider, r_max_slider]),
        HBox([log_both_check, log_rt_check, log_rz_check]),
        HBox([mask_check, threshold_slider])
        # HBox([ref_r_input])
    ])

    out = interactive_output(interactive_plotter, {
        'field_index_str': field_index_dropdown,
        'r_min': r_min_slider,
        'r_max': r_max_slider,
        'use_log_scale': log_both_check,
        'rt_use_log_scale': log_rt_check,
        'rz_use_log_scale': log_rz_check,
        'mask_low_intensity': mask_check,
        'intensity_threshold_orders': threshold_slider
        # 'ref_r': ref_r_input
    })

    display(ui, out)
else:
    print("\nNo files to display. Run the cells above to scan for data.")

In [ ]:
# output_dir = '../../output/'

# field = 'buoyancy'
# ind = 45
# t = ind*2
# index = f"{ind:03d}"
# r_min = 0
# r_max = 10.0
# use_log_scale = True

# # find the corresponding file info
# rt_info = next((f for f in data_files if f['field'] == field and f['slice_type'] == 'RTplane' and f['index'] == index), None)
# filepath_rt = os.path.join(output_dir, rt_info['filename'])
# rz_info = next((f for f in data_files if f['field'] == field and f['slice_type'] == 'RZplane' and f['index'] == index), None)
# filepath_rz = os.path.join(output_dir, rz_info['filename'])

# # --- Load RT Data ---
# rt_data = None
# rt_title = f"{field} | RTplane| t = {t}"
# if rt_info:
#     filepath_rt = os.path.join(output_dir, rt_info['filename'])
#     rt_data = load_and_reshape_data(filepath_rt, 'RTplane', grid_shapes)
#     if rt_data is None:
#         print(f"Failed to load or process data for {rt_info['filename']}.")
# else:
#     print(f"No RTplane file found for field {field}, index {index}")

# # --- Load RZ Data ---
# rz_data = None
# rz_title = f"{field} | RZplane | t = {t}"
# if rz_info:
#     filepath_rz = os.path.join(output_dir, rz_info['filename'])
#     rz_data = load_and_reshape_data(filepath_rz, 'RZplane', grid_shapes)
#     if rz_data is None:
#         print(f"Failed to load or process data for {rz_info['filename']}.")
# else:
#     print(f"No RZplane file found for field {field}, index {index}")

# fig = plt.figure(figsize=(18, 8)) # Wider figure for side-by-side
# ax1 = fig.add_subplot(1, 2, 1, projection='polar')
# ax2 = fig.add_subplot(1, 2, 2)

# # --- Plot RT Slice (or placeholder) ---
# plot_rt_slice(ax1, rt_data, r_pts, th_pts, rt_title, 
#                 r_min=r_min, r_max=r_max, 
#                 use_log_scale=use_log_scale)

# # --- Plot RZ Slice (or placeholder) ---
# plot_rz_slice(ax2, rz_data, r_pts, z_pts, rz_title, 
#                 r_min=r_min, r_max=r_max, 
#                 use_log_scale=use_log_scale)

# plt.tight_layout()
# plt.show() # Show the combined figure